# Computing Assignment 4

As usual we load the necessary libraries for this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt   # for drawing images to the screen 
import scipy.linalg as la         # for svd 

# next three lines are libraries for reading an image from 
# a url and loading it as an numpy array so we can do linear algebra on it.
import requests
from PIL import Image
from io import BytesIO

## Singular Value Decomposition (SVD) Reminders:

Recall that the SVD of a matrix $A$, with rank $r$, consists of three matrices $U$, $V$ and $\Sigma$ such that $A=U\Sigma V^T$, where $U$ and $V$ are orthogonal matrices and:

* The square roots of the non-zero eigenvalues of $A^TA$ are the singular values $\sigma_1, \ldots, \sigma_r$. $\Sigma$ is the diagonal matrix made up of these values in descending order:

$$
\Sigma = \begin{bmatrix}
\sigma_1 \vec{e}_1 & \cdots & \sigma_r \vec{e}_r & \vec{0} & \cdots & \vec{0} \\
\end{bmatrix}
$$

* The first $r$ columns of $V$ are the eigenvectors of $A^TA$ arranged in descending order of their eigenvalues, and the last $n-r$ columns are an orthonormal basis for $\operatorname{Null}(A)$:
$$
V = \begin{bmatrix}
\vec{v}_1 & \cdots & \vec{v}_r & \vec{v}_{r+1} & \cdots & \vec{v}_n \\
\end{bmatrix}
$$

* The first $r$ columns of $U$ are
$\vec{u}_i = \frac{1}{\sigma_i} A\vec{v}_i$
and the last $m-r$ columns are an orthonormal basis for $\operatorname{Null}(A^T)$:

$$
U = \begin{bmatrix}
\vec{u}_1 & \vec{u}_2 & \cdots & \vec{u}_r & \vec{u}_{r+1} & \cdots & \vec{u}_m \\
\end{bmatrix}
$$

The **key observation** we will use in our application of SVD to images is that the product $A=U\Sigma V^T$ can be decomposed into a sum:

$$
A = U\Sigma V^T = \sigma_1 \vec{u}_1 \vec{v}^T_1 + \sigma_2 \vec{u}_2 \vec{v}^T_2 + \cdots + \sigma_r \vec{u}_r \vec{v}^T_r
$$
This allows us to write $A$ as the sum of $r$ rank $1$ matrices (note a product of the form $\vec{u} \vec{v}^T$ is a matrix with rank $1$ since all columns are multiples of $\vec{u}$). 

Moreover, since we've ordered the singular values in descending order ($\sigma_1 \ge \sigma_2 \cdots \ge \sigma_r$) this means that the matrices in this sum are ordered by prominance. More prominant features of $A$ appear in the earlier terms of the sum. It is ok if this statement doesn't make sense to you now, as the point of this assignment is to come away with a visual understanding of this decomposition. 

## Python Essentials (SVD):

We use [`scipy.linalg.svd`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.svd.html) to find a singular value decomposition (SVD) in python. This function returns a 3-tuple  $(U, d, V^T )$ where 

* $d$ is a list of the non-zero singular values sorted in descending order (i.e. the diagonal entries of $\Sigma$).
* $U$ and $V$ are the full $m\times m$ and $n \times n$ matrices.

Let's try an example. We'll compute the SVD of $A = \begin{bmatrix}-1&2\\0&2\\2&1 \end{bmatrix}$.

In [ ]:
A = np.array([[-1,2],
              [0,2],
              [2,1]])

U, d, Vt = la.svd(A)

# build Sigma from list of singular values d
Sig = np.zeros(A.shape) # construct mxn zeros matrix
Sig[:d.size,:d.size] = np.diag(d) # fill diagonal with d

print(f"U = \n{U}")
print(f"Sigma = \n{Sig}")
print(f"V^T = \n{Vt}")

Let's check the product is $A$:

In [ ]:
U @ Sig @ Vt

## Images in Python

We can load an image into python as a matrix. Here we load an image of a racoon from a url and store it as an numpy array called `img`.

In [ ]:
url = 'https://www.sfu.ca/~jtmulhol/py4math/linalg/images/raccoon-1000x750.jpg'  # url of an image
response = requests.get(url)
img = Image.open(BytesIO(response.content))
img = np.array(img)  # store image as an numpy array

Let's see what size of an array the image is stored as:

In [ ]:
img.shape

This means it is a matrix with 750 rows and 1000 columns where each entry is an RGB color vector.

Let's view the image on the screen.

In [ ]:
# Display the image
plt.imshow(img)

We'll convert this picture to a gray scale image, that way we compress the RGB color vector into a single number. 
This means the gray scale image is now a $750 \times 1000$ matrix with entries between 0 (black) and 1 (white). 

In [ ]:
img_gray = img.sum(2)/255**3
plt.figure(figsize=(6, 6)) # figsize controls width and height of the diplayed image on the screen
plt.imshow(img_gray, cmap="gray")

Since the image is now stored as a big matrix we can do linear algebra on it. For example, we can ask for the rank of the matrix.

In [ ]:
np.linalg.matrix_rank(img_gray)

The rank is $750$ which is the same as the number of rows of the matrix. This is the largest rank a $750\times 1000$ matrix can have.

Now let's compute the SVD and see what we can do with it.

In [ ]:
U, sigma, Vt = la.svd(img_gray)

This factorization allows us to express the image as the sum of rank 1 images:
$$
U\Sigma V^T = \sigma_1 \vec{u}_1 \vec{v}^T_1 + \sigma_2 \vec{u}_2 \vec{v}^T_2 + \sigma_3 \vec{u}_3 \vec{v}^T_3 + \sigma_4 \vec{u}_4 \vec{v}^T_4 \cdots + \sigma_r \vec{u}_r \vec{v}^T_r
$$

The sum of the first $k$ images gives us a rank $k$ approximation of the raccoon.

Approximations to the original image:

* rank 1 approximation: $\sigma_1 \vec{u}_1 \vec{v}^T_1$
* rank 2 approximation: $\sigma_1 \vec{u}_1 \vec{v}^T_1 + \sigma_2 \vec{u}_2 \vec{v}^T_2$
* rank 3 approximation: $\sigma_1 \vec{u}_1 \vec{v}^T_1 + \sigma_2 \vec{u}_2 \vec{v}^T_2 + \sigma_3 \vec{u}_3 \vec{v}^T_3$
* etc...


Let's see what the rank 1 approximation looks like:

In [ ]:
k = 1  # number of terms in the sum for our approximation
approximate_img = np.matrix(U[:, :k]) * np.diag(sigma[:k]) * np.matrix(Vt[:k, :])
fig = plt.figure(figsize=(6,6))
plt.imshow(approximate_img, cmap='gray');

print(f"rank of approximation: {np.linalg.matrix_rank(approximate_img)}")

You can see the from this image some of the prominant features of the original image appearing as bands: the lightnees above the eyes, the darkness around the eyes.

Now change the value of $k$ to include more terms in the sum. Try changing $k$ to be values from $2$ to $50$. Recall, there are only $r = 750$ terms in the original sum (the number of non-zero singular values is equal to the rank which is 750). Therefore, $k$ should not exceed 750, and taking $k=750$ gives the original image.

In [ ]:
# YOUR CODE HERE
k = 1  # CHANGE THIS NUMBER TO VALUES BETWEEN 2 AND 50
approximate_img = np.matrix(U[:, :k]) * np.diag(sigma[:k]) * np.matrix(Vt[:k, :])
fig = plt.figure(figsize=(6,6))
plt.imshow(approximate_img, cmap='gray');

print(f"rank of approximation: {np.linalg.matrix_rank(approximate_img)}")

## <span style="color:#87CEEB">Question 1:</span>

What is the smallest value of $k$ for which you think the sum of the first $k$ terms of 
$$
U\Sigma V^T = \sigma_1 \vec{u}_1 \vec{v}^T_1 + \sigma_2 \vec{u}_2 \vec{v}^T_2 + \sigma_3 \vec{u}_3 \vec{v}^T_3 + \sigma_4 \vec{u}_4 \vec{v}^T_4 \cdots + \sigma_r \vec{u}_r \vec{v}^T_r
$$
gives a reasonably good approximation of the original image? This is just your opinion, we aren't looking for a particular value as "correct". If your value is smaller than $750$ (which hopefully it is) then this means the image can be compressed to save space. Save your answer as `k`.

In [ ]:
# YOUR CODE HERE


In [ ]:
"Verify k is an integer. (2 marks)"
assert isinstance(k,int), "k must be an integer"
assert 0 < k <= 750, "k must be between 0 and 750"
print("Problem 1 Test: Success!")

## <span style="color:#87CEEB">Question 2:</span>

What is the rank of this approximate image (as a matrix)? Save your answer as `r`.

In [ ]:
# YOUR CODE HERE


In [ ]:
"Verify r is an integer. (2 marks)"
assert isinstance(r,(int, np.integer)), "r must be an integer"
assert 0 < r <= 750, "r must be between 0 and 750"
print("Problem 2 Test: Success!")

That is it for the assignment! It was more about building an appreciation for how the tools we developed can be used in practice.